# Run Full PhoBERT Streamlit Demo on Colab

Notebook này chỉ dùng để chạy app Streamlit qua Cloudflare Tunnel. Nó **không tạo app.py**. Bạn cần đặt `app.py` ở project root trước khi chạy.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = Path('/content/drive/MyDrive/Deep/vietnamese-toxic-comment-classification')
APP_PATH = PROJECT_ROOT / 'app.py'
PHOBERT_MODEL_DIR = PROJECT_ROOT / 'outputs' / 'models' / 'note08b' / 'phobert_augmented_mixed'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PROJECT_ROOT exists:', PROJECT_ROOT.exists())
print('app.py exists:', APP_PATH.exists())
print('PHOBERT_MODEL_DIR:', PHOBERT_MODEL_DIR)
print('PHOBERT_MODEL_DIR exists:', PHOBERT_MODEL_DIR.exists())

if PROJECT_ROOT.exists():
    print('Root items:', [p.name for p in PROJECT_ROOT.iterdir()][:20])

assert PROJECT_ROOT.exists(), f'Không thấy project root: {PROJECT_ROOT}'
assert APP_PATH.exists(), f'Không thấy app.py ở project root: {APP_PATH}. Hãy upload app.py vào folder project trước.'
assert PHOBERT_MODEL_DIR.exists(), f'Không thấy PhoBERT model dir: {PHOBERT_MODEL_DIR}'
assert (PHOBERT_MODEL_DIR / 'config.json').exists(), 'Thiếu config.json trong PhoBERT model dir.'
assert ((PHOBERT_MODEL_DIR / 'model.safetensors').exists() or (PHOBERT_MODEL_DIR / 'pytorch_model.bin').exists()), 'Thiếu model weights.'

os.environ['PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['PHOBERT_MODEL_DIR'] = str(PHOBERT_MODEL_DIR)
print('OK. Ready to run Streamlit.')

## Install dependencies

In [ ]:
%cd /content/drive/MyDrive/Deep/vietnamese-toxic-comment-classification
!pip install -q streamlit transformers sentencepiece safetensors underthesea pyngrok pandas numpy scikit-learn plotly

## Check app syntax and model loading prerequisites

In [ ]:
%cd /content/drive/MyDrive/Deep/vietnamese-toxic-comment-classification
!python -m py_compile app.py

from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/Deep/vietnamese-toxic-comment-classification')
PHOBERT_MODEL_DIR = PROJECT_ROOT / 'outputs' / 'models' / 'note08b' / 'phobert_augmented_mixed'
print('config:', (PHOBERT_MODEL_DIR / 'config.json').exists())
print('model.safetensors:', (PHOBERT_MODEL_DIR / 'model.safetensors').exists())
print('pytorch_model.bin:', (PHOBERT_MODEL_DIR / 'pytorch_model.bin').exists())
print('tokenizer files:', [p.name for p in PHOBERT_MODEL_DIR.iterdir() if p.is_file()])

## Stop old Streamlit / tunnel processes

In [ ]:
!pkill -f streamlit || true
!pkill -f cloudflared || true
!sleep 2
print('Old Streamlit/cloudflared processes stopped if they existed.')

## Start Streamlit in background

In [ ]:
%cd /content/drive/MyDrive/Deep/vietnamese-toxic-comment-classification

import os
os.environ['PROJECT_ROOT'] = '/content/drive/MyDrive/Deep/vietnamese-toxic-comment-classification'
os.environ['PHOBERT_MODEL_DIR'] = '/content/drive/MyDrive/Deep/vietnamese-toxic-comment-classification/outputs/models/note08b/phobert_augmented_mixed'

!PROJECT_ROOT="$PROJECT_ROOT" PHOBERT_MODEL_DIR="$PHOBERT_MODEL_DIR" streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > /content/streamlit.log 2>&1 &
!sleep 8
!tail -n 80 /content/streamlit.log

## Install Cloudflare Tunnel

In [ ]:
from pathlib import Path
import os

cloudflared = Path('/content/cloudflared')
if not cloudflared.exists():
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared

!/content/cloudflared --version

## Open public demo URL with Cloudflare Tunnel

Cell này sẽ in URL dạng `https://...trycloudflare.com`. Mở URL đó để vào app. Nếu sau 60 giây chưa ra link, chạy cell logs ở dưới rồi chạy lại cell này.

In [ ]:
import subprocess
import time
import re

cmd = ['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8501', '--no-autoupdate']
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

public_url = None
print('Starting Cloudflare tunnel...')
for _ in range(120):
    line = proc.stdout.readline()
    if line:
        print(line, end='')
        m = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', line)
        if m:
            public_url = m.group(0)
            break
    time.sleep(0.5)

if public_url:
    print('\nAPP URL:', public_url)
else:
    print('\nKhông lấy được Cloudflare URL. Kiểm tra /content/streamlit.log hoặc chạy lại cell này.')

## Troubleshooting logs

In [ ]:
!echo '===== Streamlit log ====='
!tail -n 120 /content/streamlit.log || true
!echo '===== Processes ====='
!ps aux | grep -E 'streamlit|cloudflared' | grep -v grep || true